In [ ]:
# Figure plotting: River gauge water-level time series

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ===============================
# FILE PATH
# ===============================
csv_file = "/home/jovyan/DEA products paper/Data/410021.csv"

# ===============================
# STYLE SETTINGS
# ===============================
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.0,
    "figure.dpi": 300,
})

# ===============================
# READ WATERNSW CSV
# ===============================
df = pd.read_csv(
    csv_file,
    skiprows=4,
    header=None,
    names=[
        "DateTime",
        "Level_Metres",
        "Level_Quality",
        "Discharge_MLd",
        "Discharge_Quality",
        "Comments"
    ],
    encoding="ISO-8859-1"
)

# ===============================
# CLEAN DATA
# ===============================
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()

df["DateTime"] = pd.to_datetime(
    df["DateTime"],
    format="%H:%M:%S %d/%m/%Y",
    errors="coerce"
)

df["Level_Metres"] = pd.to_numeric(df["Level_Metres"], errors="coerce")

df = (
    df.dropna(subset=["DateTime", "Level_Metres"])
      .sort_values("DateTime")
      .copy()
)

# ===============================
# FILTER DATE RANGE
# ===============================
start_date = "1988-01-01"
end_date   = "2023-12-31"

df_plot = df[
    (df["DateTime"] >= start_date) &
    (df["DateTime"] <= end_date)
].copy()

# ===============================
# DAILY SERIES + 12-MONTH MOVING AVERAGE
# ===============================
df_plot = df_plot.set_index("DateTime")

monthly_level = df_plot["Level_Metres"].resample("MS").mean()
rolling_12m = monthly_level.rolling(window=12, center=True, min_periods=6).mean()

# ===============================
# FIGURE 1: WATER LEVEL
# ===============================
fig, ax = plt.subplots(figsize=(10, 3.6))

ax.plot(
    df_plot.index,
    df_plot["Level_Metres"],
    linewidth=0.5,
    alpha=0.35,
    color="#5DA5DA",
    label="Daily water level",
    zorder=2
)

ax.plot(
    rolling_12m.index,
    rolling_12m,
    linewidth=1.6,
    color="black",
    label="12-month moving average",
    zorder=3
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.20)

ax.set_title("(c) Water level", loc="left", fontweight="semibold", fontsize=12)
ax.set_ylabel("Water level (m)")
ax.set_xlabel("Date")
ax.set_xlim(pd.Timestamp("1988-01-01"), pd.Timestamp("2023-12-31"))

ax.xaxis.set_major_locator(mdates.YearLocator(base=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ticks = pd.date_range("1988-01-01", "2023-01-01", freq="4YS")
ax.set_xticks(ticks)

ax.legend(frameon=False, loc="upper left")

plt.tight_layout()
fig.savefig("Fig3c_WaterLevel.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved: Fig3c_WaterLevel.png")

# ===============================
# HYDROLOGICAL PERIODS
# ===============================
hydro_periods = [
    ("Wet Baseline",         "1988-01-01", "1993-12-31", "#d9f0d3"),
    ("Early Dry",            "1994-01-01", "2000-12-31", "#f0f0f0"),
    ("Millennium Drought",   "2001-01-01", "2009-12-31", "#fddbc7"),
    ("Post-Drought Floods",  "2010-01-01", "2012-12-31", "#d1e5f0"),
    ("Moderate Dry",         "2013-01-01", "2017-12-31", "#f0f0f0"),
    ("Severe Drought",       "2018-01-01", "2019-12-31", "#f4cccc"),
    ("La Niña Floods",       "2020-01-01", "2022-12-31", "#cfe2f3"),
]

# ===============================
# FIGURE 2: WATER LEVEL WITH HYDROLOGICAL PERIODS
# ===============================
fig, ax = plt.subplots(figsize=(10, 4.2))

for label, start, end, color in hydro_periods:
    ax.axvspan(
        pd.to_datetime(start),
        pd.to_datetime(end),
        color=color,
        alpha=0.18,
        zorder=0
    )

for _, start, _, _ in hydro_periods[1:]:
    ax.axvline(
        pd.to_datetime(start),
        color="grey",
        linestyle="--",
        linewidth=0.7,
        alpha=0.45,
        zorder=1
    )

ax.plot(
    df_plot.index,
    df_plot["Level_Metres"],
    linewidth=0.5,
    alpha=0.35,
    color="#5DA5DA",
    label="Daily water level",
    zorder=2
)

ax.plot(
    rolling_12m.index,
    rolling_12m,
    linewidth=1.6,
    color="black",
    label="12-month moving average",
    zorder=3
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, alpha=0.20)

ax.set_title("(c) Water level", loc="left", fontweight="semibold", fontsize=12)
ax.set_ylabel("Water level (m)")
ax.set_xlabel("Date")
ax.set_xlim(pd.Timestamp("1988-01-01"), pd.Timestamp("2023-12-31"))

ax.legend(frameon=False, loc="upper left")

label_map = {
    "Wet Baseline": "Wet Baseline",
    "Early Dry": "Early Dry",
    "Millennium Drought": "Millennium Drought",
    "Post-Drought Floods": "Post-Drought\nFloods",
    "Moderate Dry": "Moderate Dry",
    "Severe Drought": "Severe\nDrought",
    "La Niña Floods": "La Niña\nFloods",
}

for label, start, end, _ in hydro_periods:
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)
    mid_dt = start_dt + (end_dt - start_dt) / 2

    ax.text(
        mid_dt,
        -0.30,
        label_map[label],
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=8.5,
        linespacing=0.9
    )

plt.subplots_adjust(left=0.10, right=0.95, top=0.90, bottom=0.22)
fig.savefig("Fig4c_WaterLevel_Shaded.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved: Fig4c_WaterLevel_Shaded.png")